# 단일 Ticker → V10 형식 Excel Export (디버깅용)

## 사용 방법

### 전제 조건
1. **V9 DCF 노트북** (`FCFF_DCF_Valuation_v9.ipynb`) 의 Cell 1~4 가 먼저 실행되어 있어야 합니다.
   → `DCFModel`, `engine`, `clear_memory` 등이 globals 에 존재해야 함
2. **`v9_dcf_excel_export_patch.ipynb`** 의 39개 셀이 모두 실행되어 있어야 합니다.
   → `DCFModel.export_v10_excel` 메서드가 주입되어 있어야 함

### 실행 흐름
이 노트북은 단일 ticker 분석을 4단계로 나눠 실행합니다:

| 단계 | 셀 | 검증 항목 |
|---|---|---|
| Step 1 | 입력 검증 | ticker / 출력경로 / 패치 적용 여부 |
| Step 2 | DCFModel 인스턴스화 | engine 연결 |
| Step 3 | model.run() 실행 | sales/financials 로드 + FCFF 계산 + valuation |
| Step 4 | export_v10_excel | 9-sheet workbook 생성 |
| Final  | 결과 요약 | TP / Upside / Moat 출력 |

각 단계가 실패하면 그 단계만 다시 실행하면 됩니다.
중간 결과 (`_model`, `_hist_df`, `_path` 등) 는 모두 노트북 globals 에 남아있어
다른 셀에서 자유롭게 inspect 할 수 있습니다.


## Step 1 · 입력 검증

분석할 종목과 출력 경로를 정의하고, 사전 조건이 충족됐는지 확인합니다.
**여기만 수정하시면 됩니다 — 다른 셀은 그대로 실행.**


In [2]:
# ═══════════════════════════════════════════════════════════════
#  ★ 여기에 분석할 ticker 입력 ★
# ═══════════════════════════════════════════════════════════════
EXPORT_TICKER = "NFLX"          # ← 종목 변경 시 이 줄만 수정
# ═══════════════════════════════════════════════════════════════

# ── 출력 경로 (자동 분기) ──────────────────────────────────────
import os
from pathlib import Path

EXPORT_DIR = (Path(r"C:/reports") if os.name == "nt"
              else Path.home() / "reports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ── 사전 조건 검증 ─────────────────────────────────────────────
print("=" * 70)
print(f"[Step 1] 입력 검증 — {EXPORT_TICKER}")
print("=" * 70)

_errors = []

# (1) ticker 형식
if not isinstance(EXPORT_TICKER, str) or not EXPORT_TICKER.strip():
    _errors.append("EXPORT_TICKER 가 비어있음")
else:
    print(f"  ✓ Ticker:        {EXPORT_TICKER}")

# (2) 출력 경로 쓰기 가능 여부
if EXPORT_DIR.exists() and os.access(EXPORT_DIR, os.W_OK):
    print(f"  ✓ Output dir:    {EXPORT_DIR}")
else:
    _errors.append(f"출력 경로 쓰기 불가: {EXPORT_DIR}")

# (3) DCFModel 클래스 존재
if "DCFModel" in globals():
    print(f"  ✓ DCFModel:      정의됨")
else:
    _errors.append("DCFModel 이 globals 에 없음 — V9 DCF 노트북 Cell 4 를 먼저 실행하세요.")

# (4) export_v10_excel 메서드 주입 여부
if "DCFModel" in globals() and hasattr(DCFModel, "export_v10_excel"):
    print(f"  ✓ Patch 적용:    export_v10_excel 메서드 존재")
else:
    _errors.append("DCFModel.export_v10_excel 없음 — v9_dcf_excel_export_patch.ipynb 를 먼저 실행하세요.")

# (5) engine 존재
if "engine" in globals():
    print(f"  ✓ DB engine:     연결됨")
else:
    _errors.append("engine 이 globals 에 없음 — V9 DCF 노트북 Cell 3 를 먼저 실행하세요.")

# (6) clear_memory 함수 (cleanup 용)
if "clear_memory" not in globals():
    print(f"  ⚠ clear_memory:  없음 — finally 블록에서 스킵됩니다.")

# 결과
print()
if _errors:
    print("✗ 사전 조건 미충족:")
    for e in _errors:
        print(f"    · {e}")
    raise RuntimeError("Step 1 실패 — 위 메시지 확인 후 V9 노트북 / patch 노트북 먼저 실행")
else:
    print("✓ Step 1 통과 — 다음 셀로 진행하세요.")


[Step 1] 입력 검증 — NFLX
  ✓ Ticker:        NFLX
  ✓ Output dir:    C:\reports
  ⚠ clear_memory:  없음 — finally 블록에서 스킵됩니다.

✗ 사전 조건 미충족:
    · DCFModel 이 globals 에 없음 — V9 DCF 노트북 Cell 4 를 먼저 실행하세요.
    · DCFModel.export_v10_excel 없음 — v9_dcf_excel_export_patch.ipynb 를 먼저 실행하세요.
    · engine 이 globals 에 없음 — V9 DCF 노트북 Cell 3 를 먼저 실행하세요.


RuntimeError: Step 1 실패 — 위 메시지 확인 후 V9 노트북 / patch 노트북 먼저 실행

## Step 2 · `DCFModel` 인스턴스화

가벼운 단계 — DB 연결 객체를 model 에 묶기만 합니다. 실제 데이터 fetch 는 Step 3 에서.


In [ ]:
# ── DCFModel 인스턴스 생성 ─────────────────────────────────────
print(f"[Step 2] DCFModel 인스턴스화 — {EXPORT_TICKER}")

_model = DCFModel(ticker=EXPORT_TICKER, engine=engine, verbose=True)

print(f"  ✓ Instance: ticker={_model.ticker}, "
      f"horizon={_model.horizon}, verbose={_model.verbose}")
print(f"  ✓ Step 2 통과")


## Step 3 · `model.run()` 실행

가장 시간이 오래 걸리고 실패 가능성이 높은 단계. 내부적으로 4가지 작업이 순차 진행됩니다:

1. **`load_sales`** — DB `us_revenue_forecast_data` 에서 sales actual + forecast 로드
2. **`load_financials`** — FMP API 에서 income / balance / cash flow 데이터 fetch
3. **`compute_fcff`** — Sales-driven FCFF 계산 (Phase 1, 8Q forecast)
4. **`compute_valuation`** — 3-Stage DCF + Phase 2 AR(1) + Terminal Value

실패 시 어디서 났는지 traceback 으로 확인 가능. 가장 흔한 실패 원인:
- DB 에 sales forecast 가 없음 (us_revenue_forecast_notebook 먼저 실행 필요)
- FMP rate limit (429 — 잠시 후 재시도)
- BS NWC 시드 오류 (V9 의 known issue)


In [ ]:
# ── model.run() 실행 ───────────────────────────────────────────
import time
print(f"[Step 3] model.run() 실행 — {EXPORT_TICKER}")
print(f"  · load_sales / load_financials / compute_fcff / compute_valuation")
print()

_t0 = time.time()
_model.run()
_elapsed = time.time() - _t0

# ── 결과 검증 ─────────────────────────────────────────────────
import numpy as np
import pandas as pd

assert _model.result_df is not None and not _model.result_df.empty, "result_df 비어있음"
assert _model.valuation is not None, "valuation 미생성"
assert _model._fcff_history is not None, "_fcff_history 미생성"

_v = _model.valuation
print()
print(f"  ✓ result_df:       {len(_model.result_df)}Q forecast")
print(f"  ✓ _fcff_history:   {len(_model._fcff_history.dropna())}Q 과거")
print(f"  ✓ WACC:            {_v['wacc']*100:.2f}%")
print(f"  ✓ Target Price:    ${_v['target_price']:.2f}" if not np.isnan(_v['target_price']) else "  ✗ Target Price:    N/A")
print(f"  ✓ Current Price:   ${_v['current_price']:.2f}" if not np.isnan(_v['current_price']) else "  ✗ Current Price:   N/A")
print(f"  ✓ Moat:            {_v['moat_label']}  (ρ={_v.get('moat_rho', 0):.3f}, "
      f"Phase2={_v.get('n_phase2', 0)}년)")
print(f"  ✓ g₀ method:       {getattr(_model, '_eva_cache', {}).get('g0_method', 'n/a')}")
print(f"  ✓ 실행 시간:       {_elapsed:.1f}초")
print()
print("✓ Step 3 통과")


## Step 4 · `export_v10_excel` 호출 (9-sheet workbook 생성)

이 단계에서 추가 FMP fetch (10Y monthly prices + profile) 가 일어나고,
9개 시트가 순차적으로 빌드됩니다.

실패 시 어느 시트 builder 에서 났는지 traceback 으로 확인. 패치 노트북에서
해당 builder 셀만 다시 실행하면 함수 정의가 갱신되므로 이 셀을 재실행하면 됩니다.


In [ ]:
# ── Excel 생성 ─────────────────────────────────────────────────
print(f"[Step 4] export_v10_excel 호출 — {EXPORT_TICKER}")
print(f"  · 추가 FMP fetch + 9-sheet workbook 빌드")
print()

_t1 = time.time()
_path = _model.export_v10_excel(out_dir=EXPORT_DIR)
_elapsed_excel = time.time() - _t1

# ── 파일 검증 ─────────────────────────────────────────────────
assert _path.exists(), f"파일 미생성: {_path}"
_size_kb = _path.stat().st_size / 1024

# 시트 수 검증
from openpyxl import load_workbook
_wb = load_workbook(_path, read_only=True)
_n_sheets = len(_wb.sheetnames)
_wb.close()

print()
print(f"  ✓ 파일 경로:       {_path}")
print(f"  ✓ 파일 크기:       {_size_kb:.1f} KB")
print(f"  ✓ 시트 수:         {_n_sheets}개  ({", ".join(_wb.sheetnames)})")
print(f"  ✓ 빌드 시간:       {_elapsed_excel:.1f}초")
assert _n_sheets == 9, f"9개 시트 기대, {_n_sheets}개 생성됨"
print()
print("✓ Step 4 통과")


## Final · 결과 요약

원본 `cell_5_5_single_export.py` 의 출력과 동일한 형식.


In [ ]:
# ── 최종 결과 요약 ─────────────────────────────────────────────
print("=" * 70)
print(f"[Single Export] {EXPORT_TICKER} → V10 형식 Excel  (총 {_elapsed + _elapsed_excel:.1f}초)")
print("=" * 70)

_v = _model.valuation
print(f"\n  ✓ Saved: {_path}")
print(f"    Target Price : ${_v['target_price']:.2f}" if not np.isnan(_v['target_price']) else "    Target Price : N/A")
print(f"    Current Price: ${_v['current_price']:.2f}" if not np.isnan(_v['current_price']) else "    Current Price: N/A")
print(f"    Upside       : {_v['upside_pct']:.1f}%" if not np.isnan(_v.get('upside_pct', np.nan)) else "    Upside       : N/A")
print(f"    WACC         : {_v['wacc']*100:.2f}%")
print(f"    g_terminal   : {_v['g_terminal']*100:.2f}%")
print(f"    Moat         : {_v['moat_label']}  "
      f"(ρ={_v.get('moat_rho', 0):.3f}, Phase2={_v.get('n_phase2', 0)}년)")

# ── 메모리 정리 ────────────────────────────────────────────────
if "clear_memory" in globals():
    clear_memory()
    print(f"\n  ✓ clear_memory() 호출됨")


## (옵션) 디버깅용 inspection 셀

위 4단계가 모두 통과하면 아래 셀들로 중간 데이터를 자유롭게 확인할 수 있습니다.
필요한 셀만 실행하세요.


In [ ]:
# ── 1. Phase 1 (8Q forecast) FCFF 분기별 ──────────────────────
print(f"[Phase 1 — 8Q forecast]")
display(_model.result_df[["quarter", "sales_forecast", "opm_forecast",
                          "ebit", "nopat", "da", "capex", "delta_nwc",
                          "fcff"]].round(0))


In [ ]:
# ── 2. Phase 2 (AR(1)) 연도별 FCFF ────────────────────────────
import pandas as pd
print(f"[Phase 2 — AR(1) decay]  ρ={_v['moat_rho']:.3f}, {_v['n_phase2']}년")
_ph2_df = pd.DataFrame({
    "Year":   range(3, 3 + len(_v.get("ph2_annual", []))),
    "g(t)":   [f"{g*100:+.2f}%" for g in _v.get("ph2_growth", [])],
    "FCFF":   [f"{x:,.0f}" for x in _v.get("ph2_annual", [])],
})
display(_ph2_df)


In [ ]:
# ── 3. EVA / Moat 진단 ────────────────────────────────────────
print(f"[EVA / Moat 진단]")
_eva_cache = getattr(_model, "_eva_cache", {})
for k, v in _eva_cache.items():
    if k == "eva_series":
        print(f"  {k:18s}: ({len(v)}개 분기 시계열)")
    elif isinstance(v, float):
        print(f"  {k:18s}: {v:.4f}")
    else:
        print(f"  {k:18s}: {v}")


In [ ]:
# ── 4. 과거 FCFF 시계열 (terminal growth 진단용) ──────────────
print(f"[과거 FCFF 시계열 — 마지막 12개 분기]")
display(_model._fcff_history.dropna().tail(12).to_frame("FCFF").round(0))


In [ ]:
# ── 5. WACC 분해 (compute_wacc 의 입력값들) ───────────────────
print(f"[WACC 분해]")
print(f"  WACC (저장값):    {_v['wacc']*100:.4f}%")
print(f"  Net Debt:         ${_v['net_debt']/1e9:.2f}B")
print(f"  Enterprise Value: ${_v['enterprise_value']/1e9:.2f}B")
print(f"  Equity Value:     ${_v['equity_value']/1e9:.2f}B")
print(f"  Shares:           {_v['shares']/1e6:.1f}M")
print(f"  TV 비중:          {_v['tv_weight_pct']:.1f}%")
print(f"  PV(FCFF):         ${_v['pv_fcff']/1e9:.2f}B")
print(f"  PV(TV):           ${_v['pv_tv']/1e9:.2f}B")


In [ ]:
# ── 6. Excel 파일 다시 열기 (재검토용) ────────────────────────
print(f"[생성된 Excel 파일 정보]")
print(f"  경로: {_path}")
print(f"  크기: {_path.stat().st_size / 1024:.1f} KB")

from openpyxl import load_workbook
_wb = load_workbook(_path, read_only=True)
print(f"  시트:")
for sn in _wb.sheetnames:
    ws = _wb[sn]
    print(f"    · {sn:25s}  {ws.max_row} rows × {ws.max_column} cols")
_wb.close()
